# Part (a): Time-series preprocessing

Gap filling, transformations, and temporal features can change forecasting accuracy and the information available at prediction time. This chapter organizes controlled comparisons and a chronological forecast evaluation for the series audited in the [EDA chapter](01_time_series_eda.ipynb). PM2.5 is the primary target and receives every comparison in full. PM10 is the secondary target and reuses the same code with parameters fitted on its own training data. The other numeric variables enter the Granger tests and can serve as lagged forecast features.

Data loading, the shared chronological split, training-domain checks, the wind components, and the gap-aware training reference execute below. Tested helpers for seeded gap masks, gap filling, and a gap-aware Ljung–Box statistic are imported for the gap-filling comparison. The preprocessing comparisons, Granger tests, and forecast experiments remain unfinished prompts. No filling method has been selected and no forecast score has been measured. For each remaining technique, add theory, implementation, displayed measurements, and evidence-based interpretation.

The sections run in dependency order. The selected gap fill completes the training series that the stationarity tests, decompositions, and Granger tests require. The stationarity section fixes the transformation each Granger variable uses. The Granger section precedes forecasting because its training-interval results may guide which covariates enter the forecast model.

## Load the environment

The setup cell imports shared paths, the seed, and analysis libraries, and records package versions that affect the results. Inputs are reloaded explicitly so this notebook runs in a fresh kernel.

No notebook-wide random generator is shared. Each stochastic step creates its own generator or estimator from `RANDOM_STATE`, so a result does not depend on which cells ran before it.

In [1]:
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from IPython.display import display
from scipy import stats
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.tsa.seasonal import MSTL, STL, seasonal_decompose
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.multitest import multipletests
from arch.unitroot import PhillipsPerron
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from preprocessing import PROCESSED_DATA_DIR, RAW_DATA_DIR, RANDOM_STATE

sns.set_theme(style="whitegrid")
package_versions = {
    name: version(name)
    for name in ["numpy", "pandas", "scipy", "statsmodels", "arch", "scikit-learn", "holidays"]
}
package_versions["python"] = platform.python_version()
print(f"Random seed: {RANDOM_STATE}")
print(f"Raw data directory: {RAW_DATA_DIR}")
display(pd.Series(package_versions, name="version").to_frame())

Random seed: 42
Raw data directory: D:\HCMUS\DataMining\preprocessing\data\raw


,version
numpy,2.5.4
pandas,3.0.6
scipy,1.18.1
statsmodels,0.15.0
arch,8.0.0
scikit-learn,1.9.1
holidays,0.106
python,3.12.13


## Prepare data

Reload the checksum-verified archive independently of the [EDA chapter](01_time_series_eda.ipynb). Use its selected station, Wanshouxigong, and its common boundaries: training before 2015-03-01, validation from 2015-03-01 to before 2016-03-01, and testing from 2016-03-01 onward. Timestamps remain naive local Beijing civil time (UTC+8). All variables and both targets share this hourly grid. The helper checks below keep the source values and native gaps unchanged.

`wind_components` converts the compass label `wd` and the speed WSPM into $u = -s\sin\theta$ and $v = -s\cos\theta$, with the EDA chapter's convention: $s$ is WSPM in m/s and $\theta$ is the direction the wind blows from, clockwise from north. A zero speed gives $u=v=0$ even without a recorded label, because calm air has no direction; otherwise a missing label or speed leaves both components missing. The conversion uses each hour's own values, so computing it before the split cannot leak information between intervals.

`training_domain` reports each Granger candidate's training minimum and share of zero values among observed hours. It determines where $\log(x)$ and Box-Cox are defined in the stationarity section and supports the RAIN decision in the Granger section. The candidate list has 12 series, so the number of directed Granger pairs depends on whether RAIN is kept; `granger_pair_counts` shows both cases.

The brief sets the PACF search ceiling at $\min(10\log_{10} n, n/5)$, where $n$ is the number of observed training hours of the target. The code also caps it at $\lfloor n/2 \rfloor - 1$, the largest lag the PACF helper accepts; `lag_screening` shows whether that cap binds. The gap-aware training PACF screens each lag against a $\pm 1.96/\sqrt{n_k}$ band, where $n_k$ is the number of observed pairs $k$ hours apart. With tens of thousands of hours this band is narrow, so the screened order `p_reference` can reach the ceiling. In that case it shows dependence beyond the search range rather than a PACF cutoff. `p_reference` is therefore an upper bound for the candidate lag span, and the forecast section chooses the span on the validation interval.

The gap-aware training ACF/PACF is also the reference for the filling comparison. `chosen_filled_training` remains `None` until the gap-filling experiment selects and documents an actual method. A linear sensitivity comparison in EDA does not select that method here.

In [2]:
from preprocessing.air_quality import (
    add_timestamp, chronological_split, download_air_quality, read_stations, wind_components,
)
from preprocessing.time_series import (
    compare_filled_dependence, fill_gaps, gap_aware_ljung_box, gap_aware_pacf,
    pacf_lag_order, seeded_gap_mask,
)

SELECTED_STATION = "Wanshouxigong"
TARGET_COLUMNS = ["PM2.5", "PM10"]
COVARIATE_COLUMNS = ["SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
# u and v replace wd and WSPM in the Granger tests, as in the EDA chapter.
GRANGER_CANDIDATE_COLUMNS = TARGET_COLUMNS + [
    "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "u", "v",
]
VALIDATION_START = pd.Timestamp("2015-03-01")
TEST_START = pd.Timestamp("2016-03-01")

stations = add_timestamp(read_stations(download_air_quality(RAW_DATA_DIR)))
selected = stations.loc[stations["station"].eq(SELECTED_STATION)].set_index("timestamp")
assert selected.index.is_monotonic_increasing and selected.index.is_unique
assert selected.index.equals(pd.date_range(selected.index.min(), selected.index.max(), freq="h"))
selected = selected.asfreq("h")
selected = selected.join(wind_components(selected["wd"], selected["WSPM"]))
train, validation, test = chronological_split(
    selected, validation_start=VALIDATION_START, test_start=TEST_START,
)
native_missing_by_split = pd.DataFrame(
    {name: part[TARGET_COLUMNS + COVARIATE_COLUMNS + ["wd", "u", "v"]].isna().sum()
     for name, part in {"train": train, "validation": validation, "test": test}.items()}
)
training_domain = pd.DataFrame(
    {
        "observed_hours": train[GRANGER_CANDIDATE_COLUMNS].notna().sum(),
        "minimum": train[GRANGER_CANDIDATE_COLUMNS].min(),
        "zero_share": {c: float(train[c].dropna().eq(0).mean()) for c in GRANGER_CANDIDATE_COLUMNS},
    }
)
training_domain["log_and_box_cox_defined"] = training_domain["minimum"] > 0
granger_pair_counts = pd.Series(
    {
        "with RAIN": len(GRANGER_CANDIDATE_COLUMNS) * (len(GRANGER_CANDIDATE_COLUMNS) - 1),
        "without RAIN": (len(GRANGER_CANDIDATE_COLUMNS) - 1) * (len(GRANGER_CANDIDATE_COLUMNS) - 2),
    },
    name="directed_pairs",
)

observed_training_hours = {target: int(train[target].notna().sum()) for target in TARGET_COLUMNS}
brief_max_lag = {t: int(min(10 * np.log10(n), n / 5)) for t, n in observed_training_hours.items()}
# gap_aware_pacf requires max_lag below half the observed count.
MAX_LAG_BY_TARGET = {t: min(brief_max_lag[t], n // 2 - 1) for t, n in observed_training_hours.items()}
gap_reference = {target: gap_aware_pacf(train[target], MAX_LAG_BY_TARGET[target]) for target in TARGET_COLUMNS}
p_reference = {target: pacf_lag_order(gap_reference[target]) for target in TARGET_COLUMNS}
lag_screening = pd.DataFrame(
    {
        "observed_training_hours": observed_training_hours,
        "brief_max_lag": brief_max_lag,
        "pacf_helper_cap_binds": {t: MAX_LAG_BY_TARGET[t] < brief_max_lag[t] for t in TARGET_COLUMNS},
        "max_lag": MAX_LAG_BY_TARGET,
        "p_reference": p_reference,
        "p_at_search_ceiling": {t: p_reference[t] == MAX_LAG_BY_TARGET[t] for t in TARGET_COLUMNS},
    }
)

# Set this to the actual completed training frame only after selecting a filling
# method. The later guard reports pending status instead of inventing a result.
chosen_filled_training = None
display(native_missing_by_split)
display(training_domain)
display(granger_pair_counts.to_frame())
display(lag_screening)

,train,validation,test
PM2.5,257,227,212
PM10,179,127,178
SO2,384,112,173
NO2,437,138,179
CO,930,130,237
O3,792,109,177
TEMP,1,0,18
PRES,1,0,18
DEWP,1,0,18
RAIN,1,0,18


,observed_hours,minimum,zero_share,log_and_box_cox_defined
PM2.5,17263,3.000000,0.000000,True
PM10,17341,2.000000,0.000000,True
SO2,17136,0.285600,0.000000,True
NO2,17083,2.000000,0.000000,True
CO,16590,100.000000,0.000000,True
O3,16728,0.214200,0.000000,True
TEMP,17519,-11.000000,0.009361,False
PRES,17519,985.100000,0.000000,True
DEWP,17519,-31.300000,0.001770,False
RAIN,17519,0.000000,0.966836,False


,directed_pairs
with RAIN,132
without RAIN,110


,observed_training_hours,brief_max_lag,pacf_helper_cap_binds,max_lag,p_reference,p_at_search_ceiling
PM2.5,17263,42,False,42,39,False
PM10,17341,42,False,42,42,True


## Compare gap-filling methods

Artificial gaps provide known ground truth for comparing reconstruction methods. The brief asks for 5% artificial gaps that are missing completely at random (MCAR). `seeded_gap_mask` hides exactly 5% of a target's observed training hours. Selection ignores the observed values and never touches a native gap. Each call creates its own generator from `RANDOM_STATE`, so every method sees the same mask regardless of cell order. PM10 receives its own mask over its own observed hours, from the same seed.

Run two mask designs, and reuse each mask for every method:

- Point mask, the primary design: single hours drawn uniformly without replacement, as in the brief.
- Block mask, a sensitivity check: contiguous runs whose lengths are drawn from the target's native training gap lengths, placed at random among observed hours until 5% are hidden. Point gaps usually leave both neighbours observed, which favours interpolation. Native gaps are longer, so the method ranking may change.

`fill_gaps` defines the five methods. Each fills native and artificial gaps together, but only artificial gaps are scored.

| Method | Definition | Uses later observations |
|---|---|---|
| Forward fill | Last observed value | No |
| Linear | Straight line between the nearest observed hours on each side | Yes |
| Polynomial, order $d \in \{2, 3\}$ | Least-squares polynomial of degree $d$ fitted to the $d$ nearest observed hours on each side, evaluated at the missing hour | Yes |
| Cubic spline | One not-a-knot cubic spline through all observed training hours | Yes |

The polynomial methods are local fits by design. Pandas `interpolate(method="polynomial", order=3)` builds a spline through all points and returns the same values as `method="cubicspline"`, so it would duplicate the cubic spline row.

A method leaves an hour unfilled when it lacks the observations it needs, for example before the first observation or without $d$ observed hours on both sides. Report each method's unfillable count per mask. Score all methods on the common set of masked hours that every method fills, and report that count. MAE and RMSE are in µg/m³.

The Ljung–Box test applies to the reconstruction error $e_t = \hat{y}_t - y_t$, where $\hat{y}_t$ is the filled value and $y_t$ the hidden observation at hour $t$. Keep $e_t$ on the hourly grid and leave it missing outside the common scored set, so lag $k$ always means $k$ hours. `gap_aware_ljung_box` computes

$$Q(h) = \frac{n+2}{n}\sum_{k=1}^{h} n_k \hat{\rho}_k^2,$$

where $n$ is the number of scored errors, $n_k$ the number of scored pairs exactly $k$ hours apart, and $\hat{\rho}_k$ their pair-adjusted autocorrelation. On a complete series this equals the classical Ljung–Box statistic. The null hypothesis is that the errors are uncorrelated at lags 1 to $h$; compare $Q(h)$ with a $\chi^2_h$ distribution. Use $h = 24$ hours and $\alpha = 0.05$, and report the smallest $n_k$: the point mask yields few close pairs, so its test has less power than the block test. Correlated errors indicate systematic reconstruction errors that persist across neighbouring hours, such as forward fill during a rising episode.

Offline reconstruction accuracy alone does not establish forecast suitability. Run the comparison for PM2.5 in full. For PM10, apply the same procedure and report its errors in the same table.

In [3]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: compare errors for each mask design on the common scored set, and explain the unfillable counts. Discuss whether the ranking changes between point and block masks. Interpret the Ljung–Box results with their pair counts, and identify methods that require future observations. State the selected method and the reason for choosing it.

### Check dependence after the selected fill

Once `chosen_filled_training` contains the selected method's completed targets, this check compares it with the original gap-aware reference on identical training timestamps. It rejects altered observed values, shifted grids, or remaining gaps. The output reports the largest absolute ACF/PACF changes, `p_gap`, `p_filled`, and `same_p` for each target. A disagreement needs an explicit explanation before forecasting; the forecast section chooses the lag span on validation data either way. Do not consult test scores.

The stationarity tests, decompositions, and Granger tests need complete series, so they all use the training series completed by the selected method. Apply the same method to every Granger variable and record its filled count per variable. Hours that the method cannot fill at the interval boundaries are trimmed rather than extrapolated; report the trimmed span. Those sections must name the fill and retain the gap-aware ACF as the reference for the observed data. Pending status means the comparison has not been performed for a selected method.

In [4]:
if chosen_filled_training is None:
    chosen_fill_dependence_check = pd.DataFrame(
        {"target": TARGET_COLUMNS, "status": "pending: select and apply the notebook 02 filling method"}
    ).set_index("target")
else:
    chosen_fill_dependence_check = pd.DataFrame(
        {target: compare_filled_dependence(
            train[target], chosen_filled_training[target], MAX_LAG_BY_TARGET[target],
        ) for target in TARGET_COLUMNS}
    ).T
    if not chosen_fill_dependence_check["same_p"].all():
        print("Lag order changed: discuss filling sensitivity and validate candidate spans before forecasting.")
display(chosen_fill_dependence_check)


,status
target,
PM2.5,pending: select and apply the notebook 02 fill...
PM10,pending: select and apply the notebook 02 fill...


## Test stationarity and compare transformations

Stationarity assumptions affect how temporal dependence and transformations are interpreted. Choosing a transformation is a preprocessing decision, so run every test on the training interval of the filled series. Test both targets and every variable used in the Granger tests:

- ADF: the null hypothesis is a unit root. State the deterministic terms and the lag selection, for example `autolag="AIC"` with its maximum lag.
- KPSS: the null hypothesis is level stationarity (`regression="c"`) or trend stationarity (`"ct"`); state which. Statsmodels interpolates KPSS p-values from a table that covers only 0.01 to 0.10 and warns outside that range. Report such values as bounds, for example "< 0.01", not as exact p-values.
- Phillips–Perron: the null hypothesis is a unit root, with serial correlation handled by a long-run variance estimate; state its lag setting.

Store and display statistics, p-values, lag settings, and the significance threshold. With two years of hourly observations, these tests detect small departures, so a rejection says little about the size of a departure.

Compare first and second differencing, $\log(x)$, and Box-Cox where the domain permits them, where $x$ is an observed value of the series. Use `training_domain`: log and Box-Cox apply without an offset only where the training minimum is positive. TEMP, DEWP, $u$, and $v$ take negative values, and RAIN contains zeros. An additive offset would make the transformed shape depend on an arbitrary constant, so those variables use differencing only. Fit each Box-Cox $\lambda$ on training data and apply the same $\lambda$ to later intervals.

For each series, apply the transformations in a fixed sequence: the original, the first difference, and the second difference, and the same sequence after log or Box-Cox where defined. Retest after each step and stop at the first version where ADF and PP reject their null and KPSS does not. Retain the full table, including conflicting or inconclusive outcomes, and record the version chosen for each variable. Avoid differencing again once the tests agree, because over-differencing introduces negative autocorrelation. The Granger section uses the chosen versions.

In [5]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: explain which hypotheses are rejected for each series and transformation. Discuss disagreement between tests, domain constraints, and the evidence for the selected transformation without claiming that a p-value proves stationarity.


## Compare seasonal decompositions

Decomposition separates components used to describe variation and construct candidate features. The EDA chapter's hour-of-day profiles and its ACF at 24 hours support a daily period, $m = 24$, for hourly data. Weekly structure ($m = 168$) appears weaker in the EDA. The two training years give only two annual cycles, the minimum `seasonal_decompose` accepts for $m = 8760$, so an annual component would rest on two values per hour of the year. Slow annual variation therefore remains in the trend.

Compare additive and multiplicative `seasonal_decompose` and STL with $m = 24$ for both targets. Multiplicative decomposition requires positive observations, which `training_domain` confirms for both targets. Add MSTL with periods 24 and 168 to test whether a weekly component reduces the residual variance enough to justify it. State the STL settings, such as `seasonal`, `trend`, and `robust`.

Define the residual variance ratio as $\mathrm{Var}(r_t)/\mathrm{Var}(y_t)$, where $y_t$ is the observed series and $r_t$ is the residual at hour $t$. Compute both variances with denominator $n-1$ on the interval where every method has a residual: the centred moving average in `seasonal_decompose` leaves 12 hours at each end undefined. For the multiplicative model, the residual is a ratio near 1, so compare it on the log scale or report it separately rather than in the same ratio.

Save the detrended and deseasonalized training series under `PROCESSED_DATA_DIR`. Two-sided decompositions use future observations and are retrospective. A forecast feature needs a causal construction, such as an hour-of-day seasonal profile estimated on training data and subtracted at each hour.

In [6]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: compare residual variance ratios and the structure left in each residual series. Explain the seasonal-period choice, whether the weekly component earns its place, and distinguish descriptive decomposition from components available at a forecast origin.

## Construct calendar and rolling features

Calendar encodings represent recurring positions without an artificial jump at the cycle boundary. Encode hour of day, day of week, and month with $\sin(2\pi k/T)$ and $\cos(2\pi k/T)$, where $k$ is the zero-based position in the cycle and $T$ is its length: 24 for hours, 7 for weekdays, and 12 for months. The brief names weekdays and months; hour of day is added because the daily cycle dominates hourly pollution data.

Mark public holidays with `holidays.China` and record the package version from the setup cell. The calendar lists official days off only. Document any other event indicator with a source and its exact dates. Candidates for Beijing include the Spring Festival fireworks period, which extends beyond the official holiday, the municipal heating season, and the emission controls during the November 2014 APEC meeting. These are hypotheses about pollution, not verified causes.

Compute rolling mean, standard deviation, minimum, maximum, and skewness over trailing windows of 24, 168, and 720 hours. A window at forecast origin $t$ covers hours $t-w+1$ to $t$, so it excludes the target hour $t+1$. Set a minimum share of observed hours for each window and leave the statistic missing below it.

Calendar values and holiday flags are known in advance, so the feature row for target $y_{t+1}$ may encode the calendar position of hour $t+1$. Covariates such as weather and other pollutants can enter only through values observed at or before $t$; their values at $t+1$ are unknown when the forecast is made. Use the $u$ and $v$ wind components rather than the categorical direction label.

In [7]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: inspect feature values around cycle boundaries, holidays, and gaps. Show how timestamps align with the target and explain how each rolling feature excludes unavailable observations.


## Compare anomaly detectors

Different detectors flag different forms of deviation. For each target, compare Z-score on a deseasonalized series, Isolation Forest on rolling features, and STL residual thresholding. Fit thresholds and models on training data, state the Z-score threshold and the Isolation Forest `contamination` and `random_state`, store flags and detection rates, compare overlap with Jaccard similarity, and mark flagged points on labeled time plots.

State whether each detector is retrospective or usable online. Treat possible explanations for anomalies as hypotheses unless external evidence verifies them.

Flags are reported, not removed. The brief asks for detection and discussion, and the hourly values may record real pollution episodes. Removing flagged hours would also change the forecast training and evaluation sets. An anomaly indicator can enter the forecast only if it is computable at the forecast origin and improves validation error.

In [8]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: compare detection rates, overlap, and the intervals each method flags. Explain sensitivity to thresholds and distinguish unusual values from confirmed data errors or real events.


## Examine Granger relationships

A variable $X$ Granger-causes $Y$ if past values of $X$ improve the prediction of $Y$ beyond what past values of $Y$ already provide, under a linear model with a fixed lag order. The null hypothesis is that all lagged coefficients of $X$ in the equation for $Y$ are zero.

Test the variables in `GRANGER_CANDIDATE_COLUMNS` on the training interval only, after the selected gap fill. These include the $u$ and $v$ wind components in place of `wd` and WSPM. Use the stationary version of each series chosen in the stationarity section, because Granger tests on non-stationary series can report spurious relationships. Exclude RAIN if its `zero_share` in `training_domain` makes a linear test uninformative, or justify keeping it. Store the final list as `GRANGER_COLUMNS` and compute $k$, its length, and the $k(k-1)$ directed pairs from it; `granger_pair_counts` shows the two possible values.

Choose the maximum lag in hours with `VAR(...).select_order` on training data, using AIC or BIC, and state the choice. Compare pairwise tests with conditional tests from a VAR fitted to all series (`VARResults.test_causality`). Weather drives several pollutants at once, so a pairwise relationship can reflect a shared driver that the conditional test controls for. Correct p-values for multiple testing with `multipletests`, for example the Benjamini–Hochberg procedure, and state the method and threshold. With tens of thousands of hourly observations, very small effects reach significance, so report an effect size, such as the F statistic or the reduction in forecast error, alongside each p-value.

Present the results as a labeled matrix and a directed graph. The PM2.5–PM10 pair needs separate care: PM10 includes PM2.5 by definition, and Granger tests see lagged effects only, not their same-hour association.

This extension is marked advanced in the brief and also appears in the marking rubric. Predictive Granger relationships do not establish real-world causation.

In [9]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: explain supported directions under the stated lag and multiple-testing policy, and compare pairwise with conditional results. Discuss effect sizes, shared weather drivers, the PM2.5–PM10 relationship, stationarity, and the limits of a causal interpretation; state explicitly if the extension is not evaluated.


## Evaluate one-step Random Forest forecasting

A forecast comparison tests whether the chosen preprocessing supports prediction beyond a persistence baseline. Define the feature row at forecast origin $t$ and its target $y_{t+1}$, the next hour's observation. Forecasts use observed history: each test prediction uses the actual observations up to $t$, not earlier predictions. Train a separate model and persistence baseline for PM2.5 and PM10.

### Lag span

`p_reference` is an upper bound for the lag span (see `lag_screening`). Compare the spans $p \in \{1, 2, 3, 6, 12, 24, p_\text{ref}\}$, each using the observations at hours $t, t-1, \ldots, t-p+1$, which are lags 1 to $p$ of the target hour $t+1$, with all other features fixed. List the exact lag offsets for the chosen span. Complete `chosen_fill_dependence_check` first and report whether the filled series changes the screened order.

### Role of each interval

- Training: fit every preprocessing parameter, including the Box-Cox $\lambda$, the seasonal profile, anomaly thresholds, and the rolling-window coverage rule, and fit each candidate model.
- Validation: choose the lag span and the covariate set by validation MAE. Fix the Random Forest settings, including `random_state=RANDOM_STATE`, and state them.
- Test: after selection, refit the model on training and validation together with the chosen configuration, keep the preprocessing parameters fitted on training, and evaluate once.

### Missing values at prediction time

Retrospective interpolation reads the value after a gap, so it must not fill inputs at validation or test time. Forward fill each input from its last observation at or before $t$ for at most a declared number of hours, such as 3, and leave longer gaps missing; `RandomForestRegressor` accepts missing values. Training inputs follow the same causal rule, so training and test features share one construction.

Evaluate only origins where $y_t$ and $y_{t+1}$ are both observed, so the persistence forecast $\hat{y}_{t+1} = y_t$ is defined and the target is known. The model and persistence use the same origins. Report MAE and RMSE in µg/m³, the number of evaluated origins, and the number excluded for each target. Each target has its own evaluation set, so compare the model with its own baseline rather than the two targets' scores directly.

Combine lags, shifted rolling statistics, calendar features, and causal decomposition components. Invert target transformations using only known history, for example by adding the predicted difference to the observed $y_t$. Covariate lags are allowed under the availability rule in the feature section. If the Granger results guide which covariates enter the model, compare that covariate set with the full set on the validation interval.

In [10]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: for each target, compare model and baseline errors on the same test origins. Explain the validation choice of lag span and covariates, the evaluation count, feature availability, inverse transformation, and the observed-history policy before attributing any improvement to preprocessing, and discuss where the two targets differ.